In [ ]:
#1. Neo4J AuraDB 환경 설정

In [2]:
import os
from dotenv import load_dotenv

# 환경 변수 로드
load_dotenv()

True

In [3]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(
    url=os.getenv("NEO4J_URI"),
    username=os.getenv("NEO4J_USERNAME"),  
    password=os.getenv("NEO4J_PASSWORD"),
        database="neo4j-inflearn"

) 

In [3]:
# 테스트 쿼리 실행 
cypher_query = """
MATCH (n:Movie)
RETURN COUNT(n) AS Movie_Count
"""

graph.query(cypher_query)

[{'Movie_Count': 4803}]

In [4]:
# 평점 기준 상위 10개 영화를 조회하는 Cypher 쿼리
cypher_query = """
MATCH (m:Movie)                                 // Movie 라벨을 가진 모든 노드 매칭
WHERE m.rating IS NOT NULL                      // 평점 값이 존재하는 영화만 필터링
RETURN m.title AS Movie,                        // 영화 제목을 'Movie'라는 별칭으로 반환
       m.released AS Released,                  // 개봉일을 'Released'라는 별칭으로 반환
       m.rating AS Rating                       // 평점을 'Rating'이라는 별칭으로 반환
ORDER BY m.rating DESC                          // 평점 기준 내림차순 정렬 (높은 평점부터)
LIMIT 10                                        // 상위 10개 결과만 반환
"""

# Neo4j 데이터베이스에 쿼리 실행 및 결과 반환
result = graph.query(cypher_query)

result

[{'Movie': 'Stiff Upper Lips', 'Released': '1998-06-12', 'Rating': 10.0},
 {'Movie': 'Little Big Top', 'Released': '2006-01-01', 'Rating': 10.0},
 {'Movie': 'Me You and Five Bucks', 'Released': '2015-07-07', 'Rating': 10.0},
 {'Movie': 'Dancer, Texas Pop. 81', 'Released': '1998-05-01', 'Rating': 10.0},
 {'Movie': 'Sardaarji', 'Released': '2015-06-26', 'Rating': 9.5},
 {'Movie': "One Man's Hero", 'Released': '1999-08-02', 'Rating': 9.3},
 {'Movie': 'The Shawshank Redemption',
  'Released': '1994-09-23',
  'Rating': 8.5},
 {'Movie': 'There Goes My Baby', 'Released': '1994-09-02', 'Rating': 8.5},
 {'Movie': 'The Prisoner of Zenda', 'Released': '1937-09-03', 'Rating': 8.4},
 {'Movie': 'The Godfather', 'Released': '1972-03-14', 'Rating': 8.4}]

In [5]:
# 데이터프레임 변환
import pandas as pd

pd.DataFrame(result)

,Movie,Released,Rating
0,Stiff Upper Lips,1998-06-12,10.0
1,Little Big Top,2006-01-01,10.0
2,Me You and Five Bucks,2015-07-07,10.0
3,"Dancer, Texas Pop. 81",1998-05-01,10.0
4,Sardaarji,2015-06-26,9.5
5,One Man's Hero,1999-08-02,9.3
6,The Shawshank Redemption,1994-09-23,8.5
7,There Goes My Baby,1994-09-02,8.5
8,The Prisoner of Zenda,1937-09-03,8.4
9,The Godfather,1972-03-14,8.4


In [6]:
# 출연 영화가 많은 배우 상위 10명을 조회하는 Cypher 쿼리
cypher_query = """
MATCH (p:Person)-[:ACTED_IN]->(m:Movie)         // Person 노드(배우)에서 ACTED_IN 관계를 통해 Movie 노드로 연결된 패턴 찾기
                                               // p 변수는 Person 타입의 노드를 참조하고, m 변수는 Movie 타입의 노드를 참조함
RETURN p.name AS Actor,                         // 배우 이름(p.name 속성)을 'Actor'라는 별칭으로 결과에 포함
       count(m) AS MovieCount                   // 각 배우별 출연한 영화 수를 집계 함수 count()로 계산하여 'MovieCount'라는 별칭으로 반환
                                               // count(m)은 각 배우(p)가 ACTED_IN 관계로 연결된 영화(m) 노드의 개수를 계산함
ORDER BY MovieCount DESC                        // 출연 영화 수(MovieCount) 기준 내림차순(DESC) 정렬
                                               // 가장 많은 영화에 출연한 배우가 결과의 상단에 위치하게 됨
LIMIT 10                                        // 정렬된 결과 중 상위 10개의 레코드만 반환
                                               // 즉, 가장 많은 영화에 출연한 상위 10명의 배우만 결과에 포함됨
"""

# Neo4j 데이터베이스에 쿼리 실행 및 결과 반환
graph.query(cypher_query)

[{'Actor': 'Robert De Niro', 'MovieCount': 54},
 {'Actor': 'Samuel L. Jackson', 'MovieCount': 44},
 {'Actor': 'Bruce Willis', 'MovieCount': 39},
 {'Actor': 'Matt Damon', 'MovieCount': 36},
 {'Actor': 'Morgan Freeman', 'MovieCount': 35},
 {'Actor': 'Nicolas Cage', 'MovieCount': 35},
 {'Actor': 'Brad Pitt', 'MovieCount': 32},
 {'Actor': 'Johnny Depp', 'MovieCount': 32},
 {'Actor': 'Mark Wahlberg', 'MovieCount': 31},
 {'Actor': 'Owen Wilson', 'MovieCount': 31}]

In [7]:
# 장르별 영화 수를 조회하는 Cypher 쿼리
cypher_query = """
MATCH (m:Movie)-[:IN_GENRE]->(g:Genre)          // Movie 노드에서 IN_GENRE 관계를 통해 Genre 노드로 연결된 패턴 찾기
                                               // m 변수는 Movie 타입의 노드를 참조하고, g 변수는 Genre 타입의 노드를 참조함
                                               // IN_GENRE 관계는 영화가 어떤 장르에 속하는지를 나타내는 관계임
RETURN g.name AS Genre,                         // 장르 이름(g.name 속성)을 'Genre'라는 별칭으로 결과에 포함
                                               // g.name은 장르 노드의 name 속성으로, 장르의 이름을 나타냄
       count(m) AS MovieCount                   // 각 장르별 영화 수를 집계 함수 count()로 계산하여 'MovieCount'라는 별칭으로 반환
                                               // count(m)은 각 장르(g)에 IN_GENRE 관계로 연결된 영화(m) 노드의 개수를 계산함
ORDER BY MovieCount DESC                        // 영화 수(MovieCount) 기준 내림차순(DESC) 정렬
                                               // 가장 많은 영화가 속한 장르가 결과의 상단에 위치하게 됨
"""

# Neo4j 데이터베이스에 쿼리 실행 및 결과 반환
graph.query(cypher_query)

[{'Genre': 'Drama', 'MovieCount': 2297},
 {'Genre': 'Comedy', 'MovieCount': 1722},
 {'Genre': 'Thriller', 'MovieCount': 1274},
 {'Genre': 'Action', 'MovieCount': 1154},
 {'Genre': 'Romance', 'MovieCount': 894},
 {'Genre': 'Adventure', 'MovieCount': 790},
 {'Genre': 'Crime', 'MovieCount': 696},
 {'Genre': 'Science Fiction', 'MovieCount': 535},
 {'Genre': 'Horror', 'MovieCount': 519},
 {'Genre': 'Family', 'MovieCount': 513},
 {'Genre': 'Fantasy', 'MovieCount': 424},
 {'Genre': 'Mystery', 'MovieCount': 348},
 {'Genre': 'Animation', 'MovieCount': 234},
 {'Genre': 'History', 'MovieCount': 197},
 {'Genre': 'Music', 'MovieCount': 185},
 {'Genre': 'War', 'MovieCount': 144},
 {'Genre': 'Documentary', 'MovieCount': 110},
 {'Genre': 'Western', 'MovieCount': 82},
 {'Genre': 'Foreign', 'MovieCount': 34},
 {'Genre': 'TV Movie', 'MovieCount': 8}]

In [9]:
cypher_query = """
MATCH (actor:Person {name:$actor_name})-[:ACTED_IN]->(movie:Movie)<-[:ACTED_IN]-(co_actor:Person)
where co_actor<> actor
RETURN co_actor.name AS CoActor, count(movie) AS MovieCount
ORDER BY MovieCount DESC
LIMIT 10
"""
graph.query(cypher_query, params={"actor_name": "Tom Hanks"})

[{'CoActor': 'Tim Allen', 'MovieCount': 3},
 {'CoActor': 'Gary Sinise', 'MovieCount': 2},
 {'CoActor': 'Don Rickles', 'MovieCount': 2},
 {'CoActor': 'Martin Sheen', 'MovieCount': 2},
 {'CoActor': 'Joan Cusack', 'MovieCount': 2},
 {'CoActor': 'Julia Roberts', 'MovieCount': 2},
 {'CoActor': 'Elizabeth Perkins', 'MovieCount': 1},
 {'CoActor': 'Robert Loggia', 'MovieCount': 1},
 {'CoActor': 'Eugene Levy', 'MovieCount': 1},
 {'CoActor': 'Jon Lovitz', 'MovieCount': 1}]

In [10]:
cypher_query = """
MATCH(actor:Person)-[:ACTED_IN]->(m:Movie)<-[:DIRECTED]-(d: Person)
WHERE actor.name = $actor_name
RETURN d.name AS Director, count(m) AS CollaborativeMovies
ORDER BY CollaborativeMovies DESC
LIMIT 10
"""
graph.query(cypher_query, params={"actor_name": "Brad Pitt"})

[{'Director': 'David Fincher', 'CollaborativeMovies': 3},
 {'Director': 'Steven Soderbergh', 'CollaborativeMovies': 3},
 {'Director': 'Andrew Dominik', 'CollaborativeMovies': 2},
 {'Director': 'Edward Zwick', 'CollaborativeMovies': 1},
 {'Director': 'Alan J. Pakula', 'CollaborativeMovies': 1},
 {'Director': 'Jean-Jacques Annaud', 'CollaborativeMovies': 1},
 {'Director': 'Barry Levinson', 'CollaborativeMovies': 1},
 {'Director': 'Guy Ritchie', 'CollaborativeMovies': 1},
 {'Director': 'Gore Verbinski', 'CollaborativeMovies': 1},
 {'Director': 'Martin Brest', 'CollaborativeMovies': 1}]